In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Dataset folder
dataset_path = "/content/drive/MyDrive/ScreenDefectDataset"

# Prepare lists for paths and labels
all_image_paths = []
all_labels = []

# Iterate through files to create paths and labels consistently
for filename in os.listdir(dataset_path):
    path = os.path.join(dataset_path, filename)
    if os.path.isfile(path): # Ensure it's a file, not a subdirectory
        if "_ok_" in filename:
            all_image_paths.append(path)
            all_labels.append(0)  # normal
        elif "_ng_" in filename:
            all_image_paths.append(path)
            all_labels.append(1)  # defect

image_paths = all_image_paths
labels = np.array(all_labels)

print(f"Found {len(image_paths)} images and {len(labels)} corresponding labels.")

In [ ]:
# Train / Validation split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42
)

IMG_SIZE = (224,224)
BATCH_SIZE = 32

# Image loader function
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0
    return img, label

# Create datasets
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.map(lambda x,y: load_image(x,y)).batch(BATCH_SIZE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(lambda x,y: load_image(x,y)).batch(BATCH_SIZE)

In [ ]:
# Data augmentation
data_aug = tf.keras.Sequential([
    layers.RandomRotation(0.2),
    layers.RandomBrightness(0.2)
])

# MobileNetV2 model
base_model = MobileNetV2(input_shape=(224,224,3),
                         include_top=False,
                         weights="imagenet")

base_model.trainable = False

model = models.Sequential([
    data_aug,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(2, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Train
model.fit(train_ds, validation_data=val_ds, epochs=10)

# Fine tuning
base_model.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_ds, validation_data=val_ds, epochs=3)